# Week 4 Exercise: Python to Golang Code Generator

**A code gen tool that converts Python code to Golang, writes unit test cases, and compares performance of the Python and Golang versions.**

## Approach
1. Use frontier LLMs to convert Python code to Go
2. Generate Go unit tests for the converted code
3. Run both versions and compare execution times

In [ ]:
# Imports

import os
import io
import sys
import time
import subprocess
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
# Load API keys

load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("Groq API Key not set")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("OpenRouter API Key not set")

In [ ]:
# Setup clients using OpenAI-compatible APIs

gemini_client = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

groq_client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

openrouter_client = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1"
)

# Model configurations
GEMINI_MODEL = "gemini-2.5-flash"
GROQ_MODEL = "llama-3.3-70b-versatile"
OPENROUTER_MODEL = "qwen/qwen3-coder-30b-a3b-instruct"

In [ ]:
# Check if Go is installed

go_version = subprocess.run(["go", "version"], capture_output=True, text=True)
if go_version.returncode == 0:
    print(f"Go is installed: {go_version.stdout.strip()}")
else:
    print("Go is NOT installed. Please install Go from https://go.dev/dl/")

## System Prompts and Helper Functions

In [ ]:
# System prompts for code conversion and test generation

CONVERT_SYSTEM_PROMPT = """
Your task is to convert Python code into high performance Go (Golang) code.
Respond only with Go code. Do not provide any explanation other than occasional comments.
The Go response needs to produce an identical output in the fastest possible time.
The code should be a complete, runnable Go program with a main package and main function.
"""

TEST_SYSTEM_PROMPT = """
Your task is to write Go unit test cases for a given Go function.
The tests should be in a separate file (main_test.go) and use the standard "testing" package.
Important rules:
- Extract the core computation logic into a separate, exported function so it can be tested.
- Provide the MODIFIED main.go that exports the function AND the test file main_test.go.
- Separate the two files with the exact marker: // --- FILE: main_test.go ---
- The first part (before the marker) is the updated main.go.
- The second part (after the marker) is main_test.go.
- Include at least 3 test cases covering normal operation, edge cases, and expected output verification.
- Respond only with Go code.
"""

def user_prompt_for_conversion(python_code):
    return f"""
Convert this Python code to Go with the fastest possible implementation that produces identical output.
Respond only with Go code as a complete runnable program.

```python
{python_code}
```
"""

def user_prompt_for_tests(go_code):
    return f"""
Write Go unit tests for the core computation in this Go code.
First, refactor the code so the core logic is in an exported function that can be tested.
Then write comprehensive tests.

Separate the two files with: // --- FILE: main_test.go ---

```go
{go_code}
```
"""

In [ ]:
# Core functions: convert, generate tests, write files, compile, run

GO_PROJECT_DIR = os.path.join(os.getcwd(), "go_project")

def setup_go_project():
    """Create a Go module directory for our generated code."""
    os.makedirs(GO_PROJECT_DIR, exist_ok=True)
    mod_file = os.path.join(GO_PROJECT_DIR, "go.mod")
    if not os.path.exists(mod_file):
        subprocess.run(["go", "mod", "init", "codegen"], cwd=GO_PROJECT_DIR,
                       capture_output=True, text=True)
    return GO_PROJECT_DIR

def write_go_file(code, filename="main.go"):
    """Write Go code to a file in the project directory."""
    setup_go_project()
    filepath = os.path.join(GO_PROJECT_DIR, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(code)
    return filepath

def clean_code_response(reply):
    """Strip markdown code fences from LLM response."""
    reply = reply.replace('```go', '').replace('```golang', '').replace('```', '')
    return reply.strip()

def convert_to_go(client, model, python_code):
    """Use an LLM to convert Python code to Go."""
    messages = [
        {"role": "system", "content": CONVERT_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt_for_conversion(python_code)}
    ]
    response = client.chat.completions.create(model=model, messages=messages)
    reply = response.choices[0].message.content
    go_code = clean_code_response(reply)
    write_go_file(go_code, "main.go")
    return go_code

def generate_tests(client, model, go_code):
    """Use an LLM to generate Go unit tests, returning updated main.go + test file."""
    messages = [
        {"role": "system", "content": TEST_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt_for_tests(go_code)}
    ]
    response = client.chat.completions.create(model=model, messages=messages)
    reply = clean_code_response(response.choices[0].message.content)

    marker = "// --- FILE: main_test.go ---"
    if marker in reply:
        parts = reply.split(marker, 1)
        updated_main = parts[0].strip()
        test_code = parts[1].strip()
        write_go_file(updated_main, "main.go")
    else:
        test_code = reply

    write_go_file(test_code, "main_test.go")
    return test_code

def compile_and_run_go():
    """Build and run the Go program, return output."""
    setup_go_project()
    try:
        build = subprocess.run(
            ["go", "build", "-o", "main.exe", "."],
            cwd=GO_PROJECT_DIR, capture_output=True, text=True, timeout=60
        )
        if build.returncode != 0:
            return f"Build error:\n{build.stderr}"

        run_cmd = os.path.join(GO_PROJECT_DIR, "main.exe")
        result = subprocess.run(
            [run_cmd], cwd=GO_PROJECT_DIR,
            capture_output=True, text=True, timeout=120
        )
        return result.stdout if result.returncode == 0 else f"Runtime error:\n{result.stderr}"
    except subprocess.TimeoutExpired:
        return "Error: execution timed out"

def run_go_tests():
    """Run Go unit tests and return output."""
    setup_go_project()
    try:
        result = subprocess.run(
            ["go", "test", "-v", "."],
            cwd=GO_PROJECT_DIR, capture_output=True, text=True, timeout=120
        )
        return result.stdout + result.stderr
    except subprocess.TimeoutExpired:
        return "Error: tests timed out"

def run_python(code):
    """Execute Python code and capture stdout."""
    globals_dict = {"__builtins__": __builtins__}
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer
    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout
    return output

## Sample Python Code (Pi Calculation)

This is the same pi calculation used in the course labs — a good benchmark for performance comparison.

In [ ]:
# Sample Python code to convert — Pi calculation

pi_python = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(100_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

print(pi_python)

## Step 1: Run the Python version to get baseline performance

In [ ]:
# Run Python version
print("=== Python Execution ===")
python_output = run_python(pi_python)
print(python_output)

## Step 2: Convert Python to Go using different LLMs

In [ ]:
# Convert using Gemini
print("=== Converting Python to Go using Gemini ===")
go_code_gemini = convert_to_go(gemini_client, GEMINI_MODEL, pi_python)
print(go_code_gemini)

In [ ]:
# Run the Go version (Gemini generated)
print("=== Go Execution (Gemini) ===")
go_output_gemini = compile_and_run_go()
print(go_output_gemini)

## Step 3: Generate Go Unit Tests

In [ ]:
# Generate unit tests for the Go code
print("=== Generating Go Unit Tests ===")
test_code = generate_tests(gemini_client, GEMINI_MODEL, go_code_gemini)
print(test_code)

In [ ]:
# Run the Go unit tests
print("=== Running Go Unit Tests ===")
test_output = run_go_tests()
print(test_output)

## Step 4: Compare performance across multiple models

In [ ]:
# Full pipeline: convert, run, test, and compare across models

import re

def extract_execution_time(output):
    """Extract execution time from program output."""
    match = re.search(r'Execution Time:\s*([\d.]+)\s*seconds', output)
    return float(match.group(1)) if match else None

def full_pipeline(client, model, python_code, model_name):
    """Run the complete Python-to-Go pipeline for a given model."""
    print(f"\n{'='*60}")
    print(f"Model: {model_name} ({model})")
    print(f"{'='*60}")

    # Step 1: Convert to Go
    print("\n--- Converting Python to Go ---")
    try:
        go_code = convert_to_go(client, model, python_code)
        print("Conversion successful!")
    except Exception as e:
        print(f"Conversion failed: {e}")
        return None

    # Step 2: Compile and run Go
    print("\n--- Running Go code ---")
    go_output = compile_and_run_go()
    print(go_output)

    # Step 3: Generate and run tests
    print("\n--- Generating and running tests ---")
    try:
        generate_tests(client, model, go_code)
        test_output = run_go_tests()
        print(test_output[:500])
    except Exception as e:
        print(f"Test generation failed: {e}")

    # Extract timing
    go_time = extract_execution_time(go_output)
    return go_time

# Run Python baseline
print("=== Python Baseline ===")
python_output = run_python(pi_python)
print(python_output)
python_time = extract_execution_time(python_output)

# Test with all 3 models: Gemini, Groq, OpenRouter
results = {}

models_to_test = [
    (gemini_client, GEMINI_MODEL, "Gemini"),
    (groq_client, GROQ_MODEL, "Groq/Llama"),
    (openrouter_client, OPENROUTER_MODEL, "OpenRouter/Qwen3"),
]

for client, model, name in models_to_test:
    go_time = full_pipeline(client, model, pi_python, name)
    if go_time:
        results[name] = go_time

## Step 5: Performance Summary

In [ ]:
# Performance comparison summary

print("\n" + "="*60)
print("PERFORMANCE COMPARISON: Python vs Go")
print("="*60)
print(f"\nPython execution time: {python_time:.6f} seconds")
print(f"\nGo execution times by model:")
print("-" * 40)

for name, go_time in sorted(results.items(), key=lambda x: x[1]):
    speedup = python_time / go_time if go_time > 0 else float('inf')
    print(f"  {name:15s}: {go_time:.6f}s  ({speedup:.0f}X speedup)")

if results:
    best_model = min(results, key=results.get)
    best_time = results[best_model]
    print(f"\nBest performer: {best_model} with {python_time/best_time:.0f}X speedup over Python!")